# Intro til Machine Learning — 4: Aktiveringsfunktioner 📈

I notebook 3 var ReLU og sigmoid bare "krølning mellem lagene". Nu skal de i rampelyset,
for aktiveringsfunktionerne er dét, der overhovedet gør neurale netværk til andet end
lineær regression i festtøj.

I denne notebook:
1. **De fem store**: Sigmoid, Tanh, ReLU, Leaky ReLU og Softmax — formler, grafer og gradienter (afsnit 11)
2. **Aktiveringer i kamp**: vi træner netværk på måne-formede data og SER forskellen (afsnit 12)

> ⭐ = til de hurtige, 🐛 = find den bevidste fejl. I når ikke alt — det er meningen.

## Setup

In [ ]:
# Plottehjælperen fra GitHub (Plan B: upload filen manuelt via 📁-panelet)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/hjaelpefunktioner.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons, make_blobs

from hjaelpefunktioner import plot_beslutningsgraense

torch.manual_seed(42)
np.random.seed(42)

# 11: De fem store

Hurtig genopfriskning af *hvorfor* (opgave 8.6): lineært efter lineært er stadig lineært.
Uden noget ikke-lineært mellem lagene er selv verdens dybeste netværk bare én stor
`nn.Linear`. Aktiveringsfunktionen er den lille bøjning, der — gentaget gennem mange lag —
lader netværket tegne vilkårligt krøllede sammenhænge.

Her er familien:

| Funktion | Formel | Output-interval | Typisk brug |
|---|---|---|---|
| **Sigmoid** | $\sigma(z) = \dfrac{1}{1+e^{-z}}$ | $(0, 1)$ | output ved **binær klassifikation** |
| **Tanh** | $\tanh(z)$ | $(-1, 1)$ | som sigmoid, men centreret om 0 |
| **ReLU** | $\max(0, z)$ | $[0, \infty)$ | **standardvalget i skjulte lag** |
| **Leaky ReLU** | $z$ hvis $z>0$, ellers $0{,}01 z$ | $(-\infty, \infty)$ | ReLU med "lækage" — mere om hvorfor nedenfor |
| **Softmax** | $\dfrac{e^{z_i}}{\sum_j e^{z_j}}$ | sandsynligheder, sum = 1 | output ved **klassifikation med flere klasser** |

Lad os *se* dem. Her er en plotteskabelon — `torch.linspace` laver 200 jævnt fordelte
tal fra −5 til 5, og så plotter vi funktionen af dem:

In [ ]:
x = torch.linspace(-5, 5, 200)

plt.plot(x, torch.sigmoid(x))
plt.title("Sigmoid")
plt.xlabel("z")
plt.grid(True)
plt.show()

(De andre hedder `torch.tanh(x)`, `torch.relu(x)` og
`nn.functional.leaky_relu(x)` — dem skal I selv plotte i opgave 11.1.)

## Gradienterne — funktionernes skjulte personlighed

Under træning flyder gradienter **baglæns** gennem netværket (autograd!), og de skal
*igennem* hver aktiveringsfunktion undervejs. Funktionens hældning bestemmer, hvor meget
gradient der slipper igennem — og her gemmer der sig to berømte problemer.

Vi kan bruge autograd til at plotte en funktions gradient uden at kende formlen for den:

In [ ]:
x = torch.linspace(-5, 5, 200, requires_grad=True)
y = torch.sigmoid(x)
y.sum().backward()          # giver gradienten i alle 200 punkter på én gang

plt.plot(x.detach(), y.detach(), label="sigmoid(z)")
plt.plot(x.detach(), x.grad, "--", label="gradient (hældning)")
plt.title("Sigmoid og dens gradient")
plt.legend()
plt.grid(True)
plt.show()

Se på den stiplede kurve: sigmoids gradient er **højst 0,25** — og ude i "halerne"
(store positive/negative z) er den praktisk talt **nul**. Sender man en gradient baglæns
gennem mange sigmoid-lag, ganges den med et lille tal for hvert lag og **forsvinder**:
det berømte *vanishing gradient*-problem. Netværkets forreste lag lærer aldrig noget.
(I mærker det selv i opgave 12.5.)

ReLU har det modsat: hældning præcis 1 for alle positive z (gradienten passerer urørt!) —
men hældning **0** for alle negative. En neuron, der altid får negative input, får aldrig
gradient og kan aldrig lære igen: en **død ReLU** 💀. Derfor opfandt man **Leaky ReLU**,
som lader en lille smule gradient sive igennem på den negative side — et plaster på
problemet.

## Softmax: sandsynlighedsmaskinen

De fire første funktioner arbejder på ét tal ad gangen. **Softmax** er anderledes: den
tager en hel **stribe** tal (ét "point" pr. klasse) og laver dem om til sandsynligheder,
der er positive og summer til 1:

In [ ]:
point = torch.tensor([2.0, 1.0, 0.1])            # netværkets rå point for 3 klasser
sandsynligheder = torch.softmax(point, dim=0)

print(sandsynligheder)
print("sum:", sandsynligheder.sum().item())

Klassen med flest point får den største sandsynlighed, men de andre får også en bid —
softmax siger ikke bare "vinderen tager alt", den siger *hvor sikker* den er.

### Opgaver

##### Opgave 11.1
Brug plotteskabelonen til at tegne **alle fire** af de "almindelige" funktioner — sigmoid,
tanh, ReLU og leaky ReLU — i ét 2×2-subplot-grid (genbrug subplot-tricket fra opgave 4.8).
Aflæs for hver: hvad er output-intervallet?

In [ ]:
x = torch.linspace(-5, 5, 200)

fig, akser = plt.subplots(2, 2, figsize=(10, 7))
akser[0, 0].plot(x, torch.sigmoid(x))
akser[0, 0].set_title("Sigmoid")
# ... fortsæt med tanh, relu og leaky_relu (nn.functional.leaky_relu(x))
plt.tight_layout()
plt.show()

##### Opgave 11.2
Byg selv sigmoid ud af `torch.exp`: udfyld formlen $\sigma(z) = \dfrac{1}{1+e^{-z}}$ og
tjek, at jeres udgave giver det samme som `torch.sigmoid`.

In [ ]:
def min_sigmoid(z):
    return 1 / (1 + ...)

z = torch.tensor([-2.0, 0.0, 3.0])
print("min:    ", min_sigmoid(z))
print("PyTorch:", torch.sigmoid(z))

##### Opgave 11.3
Byg selv ReLU. Udfyld med **enten** `torch.clamp(z, min=...)` (som "klipper" tal af ved en
grænse) **eller** `torch.where(betingelse, hvis_sand, hvis_falsk)` — og tjek mod PyTorch.

In [ ]:
def min_relu(z):
    return ...

z = torch.tensor([-3.0, -0.5, 0.0, 2.0])
print("min:    ", min_relu(z))
print("PyTorch:", torch.relu(z))

##### Opgave 11.4
Genbrug gradient-plotteskabelonen (fra teorien ovenfor) på sigmoid, og aflæs: hvad er
gradienten cirka ved $z = \pm 5$? Forestil jer nu en gradient, der skal baglæns gennem
**10 sigmoid-lag**, hvor den hver gang ganges med et tal af den størrelse — hvad sker der
med den, og hvorfor er det et problem for læring?

*Skriv jeres svar her:* $\dots$

##### Opgave 11.5
Leaky ReLU har en indstillelig "lækage": `nn.functional.leaky_relu(x, negative_slope=...)`.
Plot den med `negative_slope` på **0.01, 0.1 og 0.5** i samme figur (med labels og legend).
Hvornår holder den op med at ligne ReLU og begynder bare at ligne en ret linje?

In [ ]:
x = torch.linspace(-5, 5, 200)
plt.plot(x, nn.functional.leaky_relu(x, negative_slope=0.01), label="0.01")
# ← tilføj 0.1 og 0.5
plt.legend()
plt.grid(True)
plt.show()

##### Opgave 11.6
Udfyld softmax-kaldet (hvilken `dim`?), og tjek at summen er 1. Gang derefter alle point
med 10 (`point * 10`) og kør igen — hvad sker der med sandsynlighederne, og hvorfor giver
det mening?

In [ ]:
point = torch.tensor([2.0, 1.0, 0.1])
sandsynligheder = torch.softmax(point, dim=...)
print(sandsynligheder, "| sum:", sandsynligheder.sum().item())

##### Opgave 11.7 🐛
Her er softmax brugt på et batch med ét eksempel og tre klasser — men ALLE
"sandsynligheder" bliver 1.0?! Kig på tensorens shape og på `dim`-argumentet: hvilken
retning bliver der normaliseret langs? Ret `dim`, så det giver mening.

In [ ]:
point = torch.tensor([[2.0, 1.0, 0.1]])     # shape (1, 3): 1 eksempel, 3 klasser
sandsynligheder = torch.softmax(point, dim=0)
print(sandsynligheder)                        # [[1., 1., 1.]] ... det er vist ikke sandsynligheder

##### Opgave 11.8
Vælg aktiveringsfunktion til hver situation — og begrund kort:

(a) de skjulte lag i et netværk,
(b) output-laget ved binær klassifikation (spam/ikke spam),
(c) output-laget ved klassifikation med 10 klasser (håndskrevne cifre!),
(d) output-laget ved regression (forudsig en huspris).

*Skriv jeres svar her:* $\dots$

##### Opgave 11.9 ⭐
Plot **gradienterne** af sigmoid, tanh, ReLU og leaky ReLU i ét samlet plot (genbrug
autograd-tricket — én funktion ad gangen, samme `x`). Hvilken funktion har den "sundeste"
gradient for store positive $z$ — og hvad med for store negative?

In [ ]:
funktioner = {
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "ReLU": torch.relu,
    "leaky ReLU": nn.functional.leaky_relu,
}

for navn, f in funktioner.items():
    x = torch.linspace(-5, 5, 200, requires_grad=True)
    f(x).sum().backward()
    plt.plot(x.detach(), x.grad, label=navn)   # x.grad ER gradienten

plt.legend()
plt.title("Gradienter")
plt.grid(True)
plt.show()

# 12: Aktiveringer i kamp — månedata 🌙

Nu skal påstandene testes. `make_moons` laver et syntetisk 2D-datasæt: to klasser formet
som halvmåner, der griber ind i hinanden. Det er perfekt til formålet, for med kun 2
features kan vi **tegne alt**, inklusive hvad netværket tænker:

In [ ]:
X_np, y_np = make_moons(n_samples=400, noise=0.2, random_state=42)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu_r", edgecolors="black", s=25)
plt.title("To måner — kan et netværk skille dem ad?")
plt.show()

Ingen ret linje kan skille de to måner ad — prøv selv at "tegne" en med øjnene.
Spørgsmålet er, om et netværk kan tegne noget bedre.

Vi genbruger træningsopskriften fra notebook 3 og pakker den i en funktion (så vi ikke
skal skrive de samme 10 linjer 10 gange — det er jo derfor, funktioner findes). Læs den
igennem: det er PRÆCIS de fem trin, I kender:

In [ ]:
def traen(model, X, y, epoker=1500, lr=0.01):
    tabsfunktion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    historik = []
    for epoke in range(epoker):
        optimizer.zero_grad()                          # 1. nulstil
        y_hat = model(X).squeeze()                     # 2. forward
        tab = tabsfunktion(y_hat, y)                   # 3. tab
        tab.backward()                                 # 4. backward
        optimizer.step()                               # 5. step
        historik.append(tab.item())
    return historik

In [ ]:
class MaaneNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(2, 16)
        self.lag2 = nn.Linear(16, 16)
        self.lag3 = nn.Linear(16, 1)
        self.aktivering = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.aktivering(self.lag1(x))
        x = self.aktivering(self.lag2(x))
        x = self.sigmoid(self.lag3(x))
        return x

model = MaaneNet()
historik = traen(model, X, y)
print("sluttab:", round(historik[-1], 4))

Og nu det fede: `plot_beslutningsgraense` (fra hjælpefilen) farver **hele planen**
efter, hvad modellen ville svare i hvert punkt. Den sorte streg er netværkets
**beslutningsgrænse** — skillelinjen mellem "klasse 0" og "klasse 1":

In [ ]:
plot_beslutningsgraense(model, X, y, titel="MaaneNet med ReLU")

Se den kurve! Netværket har selv opfundet en bugtet grænse, der følger månerne.
DET er, hvad aktiveringsfunktioner køber os. Men påstanden var jo, at det ikke kan lade
sig gøre *uden* dem — det tjekker I selv i opgave 12.1. 😏

### Opgaver

##### Opgave 12.1
Klassen nedenfor er MaaneNet **uden aktivering mellem lagene** (sigmoiden til sidst er kun
for at få en sandsynlighed ud). Træn den og plot dens beslutningsgrænse. Sammenlign med
ReLU-udgaven ovenfor: hvad er grænsen reduceret til — og hvorfor kan den ALDRIG lære
månerne, uanset træningstid? (Opgave 8.6 har svaret.)

In [ ]:
class LineaertNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(2, 16)
        self.lag2 = nn.Linear(16, 16)
        self.lag3 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.lag1(x)          # ingen aktivering!
        x = self.lag2(x)          # ingen aktivering!
        x = self.sigmoid(self.lag3(x))
        return x

model_lin = LineaertNet()
historik = traen(model_lin, X, y)
print("sluttab:", round(historik[-1], 4))
plot_beslutningsgraense(model_lin, X, y, titel="Uden aktivering")

##### Opgave 12.2
Byt ReLU ud med **Sigmoid**, **Tanh** og **LeakyReLU** (én ad gangen — skabelonen tager
aktiveringen som parameter) og plot de fire tabskurver i samme figur. Hvem lærer hurtigst
på månerne?

In [ ]:
class FleksNet(nn.Module):
    def __init__(self, aktivering):
        super().__init__()
        self.lag1 = nn.Linear(2, 16)
        self.lag2 = nn.Linear(16, 16)
        self.lag3 = nn.Linear(16, 1)
        self.aktivering = aktivering
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.aktivering(self.lag1(x))
        x = self.aktivering(self.lag2(x))
        x = self.sigmoid(self.lag3(x))
        return x

historik = traen(FleksNet(nn.ReLU()), X, y)
plt.plot(historik, label="ReLU")
# ← tilføj nn.Sigmoid(), nn.Tanh() og nn.LeakyReLU()
plt.legend()
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

##### Opgave 12.3
Skru op for støjen i `make_moons` (prøv `noise=0.3`, `0.5` og `0.8`), gentræn og plot
beslutningsgrænsen hver gang. Hvornår "knækker" modellen — og hvordan ser grænsen ud, når
den prøver at lære rent kaos?

In [ ]:
X_np2, y_np2 = make_moons(n_samples=400, noise=0.3, random_state=42)   # ← skru på noise
X2 = torch.tensor(X_np2, dtype=torch.float32)
y2 = torch.tensor(y_np2, dtype=torch.float32)

model_stoej = MaaneNet()
traen(model_stoej, X2, y2)
plot_beslutningsgraense(model_stoej, X2, y2, titel="noise = 0.3")

##### Opgave 12.4 🐛
Netværket nedenfor skal lave **regression**: forudsige $y \approx 2x + 1$ (værdier fra 1
til 11). Men forudsigelserne klistrer fast lige over 1, og tabet er enormt. Kig på
`forward` — der er sneget sig et lag ind, som IKKE hører hjemme i en regressionsmodel
(opgave 11.8d!). Fjern det og se forskellen.

In [ ]:
x_reg = torch.linspace(0, 5, 100).reshape(-1, 1)
y_reg = 2 * x_reg.squeeze() + 1 + torch.randn(100) * 0.3

class RegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(1, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.aktivering(self.lag1(x))
        x = self.sigmoid(self.lag2(x))    # hmm...
        return x

model_reg = RegNet()
tabsfunktion = nn.MSELoss()
optimizer = torch.optim.Adam(model_reg.parameters(), lr=0.01)
for epoke in range(1000):
    optimizer.zero_grad()
    tab = tabsfunktion(model_reg(x_reg).squeeze(), y_reg)
    tab.backward()
    optimizer.step()

with torch.no_grad():
    plt.scatter(x_reg, y_reg, alpha=0.4)
    plt.plot(x_reg, model_reg(x_reg).detach(), color="crimson", linewidth=2)
plt.title(f"sluttab: {tab.item():.2f}")
plt.show()

##### Opgave 12.5 ⭐
Vanishing gradients LIVE: skabelonen bygger et **6-lags** netværk, hvor aktiveringen kan
vælges. Træn én udgave med `nn.Sigmoid()` og én med `nn.ReLU()`, og plot de to tabskurver
sammen. Hvad ser I — og hvad har det med opgave 11.4 at gøre?

In [ ]:
class DybtNet(nn.Module):
    def __init__(self, aktivering):
        super().__init__()
        self.lag = nn.ModuleList([nn.Linear(2, 16)] +
                                 [nn.Linear(16, 16) for i in range(4)] +
                                 [nn.Linear(16, 1)])
        self.aktivering = aktivering
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        for lag in self.lag[:-1]:
            x = self.aktivering(lag(x))
        return self.sigmoid(self.lag[-1](x))

historik = traen(DybtNet(nn.Sigmoid()), X, y, epoker=2000)
plt.plot(historik, label="6 lag, sigmoid overalt")
# ← træn og plot også DybtNet(nn.ReLU())
plt.legend()
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

##### Opgave 12.6 ⭐
Klassifikation med **3 klasser**: `make_blobs` laver tre klumper. Ved flere klasser skal
netværket give **3 tal ud** (ét point pr. klasse), og tabsfunktionen skal være
`nn.CrossEntropyLoss`. Udfyld de tre huller.

**VIGTIG detalje** (I får brug for den i notebook 5): `nn.CrossEntropyLoss` vil have de
**RÅ point** — den kører selv softmax indeni! Derfor har modellen INGEN
aktiveringsfunktion til sidst. Og målene `y` skal være hele tal (klassenumre), ikke
kommatal.

In [ ]:
X_np3, y_np3 = make_blobs(n_samples=450, centers=3, cluster_std=1.2, random_state=42)
X3 = torch.tensor(X_np3, dtype=torch.float32)
y3 = torch.tensor(y_np3, dtype=torch.long)     # klassenumre 0, 1, 2 — hele tal!

class TreKlasseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(2, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, ...)          # ← hvor mange tal skal ud?

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))   # rå point — ingen softmax her!

model3 = TreKlasseNet()
tabsfunktion = ...                              # ← tabsfunktionen til flere klasser
optimizer = torch.optim.Adam(model3.parameters(), lr=0.01)
for epoke in range(1000):
    optimizer.zero_grad()
    tab = tabsfunktion(model3(X3), ...)         # ← hvad skal der sammenlignes med?
    tab.backward()
    optimizer.step()

print("sluttab:", round(tab.item(), 4))
plot_beslutningsgraense(model3, X3, y3, titel="3 klasser")

##### Opgave 12.7
Hvad kan et netværk med kun **ÉN skjult neuron** (2 → 1 → 1)? Træn det på månerne og plot
beslutningsgrænsen. Beskriv, hvad I ser — og forklar hvorfor grænsen ser sådan ud, når
én ReLU-neuron kun kan "knække" planen én gang.

In [ ]:
class MiniNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(2, 1)     # én enkelt, ensom neuron
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(1, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.lag2(self.aktivering(self.lag1(x))))

model_mini = MiniNet()
traen(model_mini, X, y, epoker=2000)
plot_beslutningsgraense(model_mini, X, y, titel="1 skjult neuron")

##### Opgave 12.8
Sigmoid kom først (1980'erne), ReLU tog over (2010'erne). Ud fra alt, hvad I har set i
denne notebook: giv mindst to grunde til, at ReLU i dag er standardvalget i skjulte lag —
og én situation, hvor sigmoid stadig er det rigtige valg.

*Skriv jeres svar her:* $\dots$

# Godt gået! 🎉

I kender nu de fem store aktiveringsfunktioner, deres gradienter, deres faldgruber — og I
har med egne øjne set, at et netværk uden aktivering bare er en ret linje med selvtillid.

**Sidste stop:** `5-MNIST.ipynb` — alt fra emnet samlet i én model, der lærer at læse
**håndskrevne tal**. Ja, rigtige billeder. Ja, det virker. 🔢👁️